# Exercises XP: Day 3 - BERT in Practice
Follow the prompts below. Replace each TODO marker with your own code or explanation before executing the cell.


## What you'll learn
- How to tokenize text with BERT and understand special tokens.
- How to run a pretrained sentiment pipeline.
- How to build custom BERT-based sentiment and NER analyzers.
- How to compare encoder (BERT) versus decoder (GPT) families.
- How BERT supplies retrieval power inside a RAG stack.


## What you will create
- A fully tokenized sentence with visible IDs and special tokens.
- A working sentiment pipeline powered by a fine-tuned DistilBERT model.
- Custom helper classes for sentiment classification and NER.
- A comparison table that contrasts BERT and GPT.
- A written explanation of how BERT embeddings drive retrieval in RAG.


> Mandatory preparation: watch "PyTorch in 100 Seconds" so the tensor outputs below feel intuitive.

## Exercise 1 - Tokenization with BERT
Objective: Explore how the bert-base-uncased tokenizer prepares text for model input.

Instructions:
1. (Optional) Install the required libraries.
2. Load the tokenizer, craft a sample sentence, and encode it with padding plus truncation.
3. Print the tokens next to their integer IDs and flag the special tokens.
4. Inspect the attention mask to see how padding is hidden from the model.

Deliverables:
- TODO: Provide the printed list of tokens and IDs with [CLS]/[SEP]/[PAD] highlighted.
- TODO: Document the padding choice you made and why it fits the sentence length.


In [3]:
# Optional setup: install dependencies if they are missing in your environment.
%pip install -q transformers torch


In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

sample_sentence = "Hello im a super computer from another galaxy"
print(sample_sentence)


Hello im a super computer from another galaxy


In [7]:
encoding = tokenizer(
    sample_sentence,
    add_special_tokens=True,
    padding="max_length",
    truncation=True,
    max_length=24,  # TODO: adjust if your sentence needs more room
    return_attention_mask=True,
    return_tensors="pt"
)

input_ids = encoding["input_ids"][0].tolist()
tokens = tokenizer.convert_ids_to_tokens(input_ids)
print("index | token        | id")
print("-------------------------")
for idx, (token, token_id) in enumerate(zip(tokens, input_ids)):
    print(f"{idx:>5} | {token:<12} | {token_id:>5}")

print("\nAttention mask:", encoding["attention_mask"][0].tolist())
special_positions = [(i, tok) for i, tok in enumerate(tokens) if tok in tokenizer.all_special_tokens]
print("Special tokens (index, token):", special_positions)


index | token        | id
-------------------------
    0 | [CLS]        |   101
    1 | hello        |  7592
    2 | im           | 10047
    3 | a            |  1037
    4 | super        |  3565
    5 | computer     |  3274
    6 | from         |  2013
    7 | another      |  2178
    8 | galaxy       |  9088
    9 | [SEP]        |   102
   10 | [PAD]        |     0
   11 | [PAD]        |     0
   12 | [PAD]        |     0
   13 | [PAD]        |     0
   14 | [PAD]        |     0
   15 | [PAD]        |     0
   16 | [PAD]        |     0
   17 | [PAD]        |     0
   18 | [PAD]        |     0
   19 | [PAD]        |     0
   20 | [PAD]        |     0
   21 | [PAD]        |     0
   22 | [PAD]        |     0
   23 | [PAD]        |     0

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Special tokens (index, token): [(0, '[CLS]'), (9, '[SEP]'), (10, '[PAD]'), (11, '[PAD]'), (12, '[PAD]'), (13, '[PAD]'), (14, '[PAD]'), (15, '[PAD]'), (16, '[PAD]'

### Exercise 1 reflection
- [CLS] sits at position 0 and its final hidden state is used as a sentence-level representation (e.g., for classification). [SEP] marks the end of a segment so the model knows where the sentence finishes.
- The attention mask is 1 for every real token and 0 for every [PAD] token. During self-attention, positions with mask=0 are ignored, so padding never influences the model's understanding of the real words..


## Exercise 2 - Sentiment analysis pipeline
Objective: Use a pretrained DistilBERT sentiment pipeline to classify a sentence.

Instructions:
1. Import the `pipeline` helper from transformers.
2. Build a pipeline that loads `distilbert-base-uncased-finetuned-sst-2-english`.
3. Pass in a sentence and review the predicted label and score.

Deliverables:
- TODO: Record the sentence you tested.
- TODO: Capture the label plus confidence score and interpret the result.


In [8]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

sentence = "I absolutely loved the movie, it was fantastic!"
prediction = sentiment_pipeline(sentence)
prediction


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9998804330825806}]

### Exercise 2 reflection
- Yes, the label matches expectation — the sentence is clearly positive.
- A score of ~0.9998 means the model is 99.98% confident. The score is a softmax probability between 0 and 1; the closer to 1, the more certain the model is about its label.


## Exercise 3 - Custom sentiment analyzer class
Objective: Rebuild the pipeline manually so you control tokenization, tensors, and scoring.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForSequenceClassification`.
2. Implement `BERTSentimentAnalyzer` with methods for initialization, preprocessing, and prediction.
3. Test the class with multiple sentences.

Hints:
- Keep a `max_length` attribute so you can reuse it while tokenizing.
- Apply `torch.softmax` to transform logits into probabilities.
- Return both the label and the probability for clarity.


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from typing import Dict

class BERTSentimentAnalyzer:
    def __init__(self, model_name: str = "distilbert-base-uncased-finetuned-sst-2-english", max_length: int = 128):
      self.max_length = max_length
      self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
      self.model.to(self.device)
      self.model.eval()

    def preprocess(self, text: str) -> Dict[str, torch.Tensor]:
      text = text.strip()
      tokens = self.tokenizer(
          text,
          padding="max_length",
          truncation=True,
          max_length=self.max_length,
          return_tensors="pt"
      )
      return {k: v.to(self.device) for k, v in tokens.items()}


    def predict(self, text: str) -> Dict[str, float]:
      inputs = self.preprocess(text)
      with torch.no_grad():
        outputs = self.model(**inputs)
      probs = torch.softmax(outputs.logits, dim=-1)[0]
      label_id = torch.argmax(probs).item()
      label = self.model.config.id2label[label_id]
      confidence = probs[label_id].item()
      return {"label": label, "confidence": round(confidence, 4)}



In [12]:
# TODO: instantiate your analyzer and test several sentences once the class is ready.
analyzer = BERTSentimentAnalyzer()
samples = [
   "This is the best day of my life!",
   "I hate waiting in long queues."
 ]
for text in samples:
     print(text)
     print(analyzer.predict(text))


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

This is the best day of my life!
{'label': 'POSITIVE', 'confidence': 0.9999}
I hate waiting in long queues.
{'label': 'NEGATIVE', 'confidence': 0.9968}


## Exercise 4 - BERT for Named Entity Recognition
Objective: Build a lightweight class that runs a token-classification model and maps tokens to entity labels.

Instructions:
1. Import `AutoTokenizer` and `AutoModelForTokenClassification`.
2. Implement `BERTNamedEntityRecognizer` with init plus a `recognize` method.
3. Tokenize sample text, run the model, convert the predictions to entity spans, and test with a short paragraph.

Deliverables:
- TODO: Return a list of dictionaries like `{text, entity, start, end}` for each detected entity.
- TODO: Explain how you handled subword tokens that begin with `##`.


In [15]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

class BERTNamedEntityRecognizer:
    def __init__(self, model_name: str = "dslim/bert-base-NER"):
      self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
      self.tokenizer = AutoTokenizer.from_pretrained(model_name)
      self.model = AutoModelForTokenClassification.from_pretrained(model_name)
      self.model.to(self.device)
      self.model.eval()

    def recognize(self, text: str):
      inputs = self.tokenizer(
          text,
          return_tensors="pt",
          return_offsets_mapping=True,
          truncation=True
      )
      offset_mapping = inputs.pop("offset_mapping")[0].tolist()
      inputs = {k: v.to(self.device) for k, v in inputs.items()}

      with torch.no_grad():
        outputs = self.model(**inputs)

      predictions = torch.argmax(outputs.logits, dim=-1)[0].tolist()
      tokens = self.tokenizer.convert_ids_to_tokens(inputs["input_ids"][0].tolist())

      entities = []
      current_entity = None

      for token, pred_id, offsets in zip(tokens, predictions, offset_mapping):
        label = self.model.config.id2label[pred_id]

        if token in self.tokenizer.all_special_tokens:
          continue

        if token.startswith("##"):
          if current_entity:
            current_entity["text"] += token[2:]
            current_entity["end"] = offsets[1]
          continue

        if label.startswith("B-"):
          if current_entity:
            entities.append(current_entity)
          current_entity = {
            "text": token,
            "entity": label[2:],
            "start": offsets[0],
            "end": offsets[1]
          }
        elif label.startswith("I-") and current_entity:  # inside an entity
          current_entity["text"] += " " + token
          current_entity["end"] = offsets[1]
        else:  # "O" — not an entity
          if current_entity:
            entities.append(current_entity)
          current_entity = None

      if current_entity:
        entities.append(current_entity)

      return entities

In [16]:
# TODO: instantiate the recognizer and test it on text that includes people, places, or organizations.
ner = BERTNamedEntityRecognizer()
sample_text = "Elon Musk founded SpaceX in California and later moved to Texas."
ner.recognize(sample_text)


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'text': 'Elon Musk', 'entity': 'PER', 'start': 0, 'end': 9},
 {'text': 'SpaceX', 'entity': 'ORG', 'start': 18, 'end': 24},
 {'text': 'California', 'entity': 'LOC', 'start': 28, 'end': 38},
 {'text': 'Texas', 'entity': 'LOC', 'start': 58, 'end': 63}]

## Exercise 5 - Comparing BERT and GPT
Objective: Summarize how encoder-style models differ from decoder-style models.

Fill the table with concise statements (one line each).

| Category | BERT | GPT |
|----------|------|-----|
| Architecture | Encoder-only; reads the full sequence bidirectionally | Decoder-only; reads left-to-right (autoregressive) |
| Primary purpose | Understanding and representing text | Generating new text |
| Typical use cases | Sentiment analysis, NER, question answering, text classification | Chatbots, text completion, summarization, code generation |
| Strengths | Rich contextual embeddings; sees both left and right context simultaneously | Excellent at fluent, coherent long-form generation |
| Weaknesses | Cannot generate text natively; needs fine-tuning per task | Weaker at precise classification; no bidirectional context |


## Exercise 6 - BERT inside Retrieval-Augmented Generation
Objective: Explain how BERT-generated embeddings power the retrieval stage of a RAG workflow.

Address each bullet with a short paragraph:
1. Describe how BERT encodes queries and documents: BERT processes text bidirectionally, meaning each token attends to every other token in the sequence. The final hidden state of the [CLS] token (or a mean-pool of all token states) produces a dense vector — called an embedding — that captures the semantic meaning of the entire input. Both the user's query and every document in the knowledge base get converted into these fixed-size vectors.

2. Explain how those embeddings are stored and searched in a vector database: Document embeddings are pre-computed and stored in a vector database like Pinecone, FAISS, or Weaviate. At query time, BERT encodes the user's question into a vector and the database performs approximate nearest-neighbor (ANN) search — finding the stored document vectors that are most geometrically similar (usually measured by cosine similarity). This is extremely fast even over millions of documents.

3. Outline how the retrieved passages are handed to a generative model like GPT: The top-k most similar documents are retrieved and their raw text is concatenated with the original query into a prompt like: "Context: [passage 1] [passage 2]... Question: [query] Answer:". This combined prompt is then fed to a generative decoder model (like GPT) which produces an answer grounded in the retrieved information, reducing hallucination.

4. Provide a concrete application example (industry or product) where RAG with BERT makes sense: A customer support chatbot for a software company is a great fit. The company's entire documentation, FAQs, and ticket history are embedded with BERT and stored in a vector DB. When a user asks "How do I reset my API key?", BERT encodes the query, retrieves the 3 most relevant doc sections, and GPT generates a precise, friendly answer from them — without needing to retrain the model every time docs are updated.
